[EN]

# 02 — NVT-MD of the Interface

Run Langevin NVT-MD of the interface built in `01_build_interface.ipynb` (default: 100,000 steps, 300 K, 1 fs/step → 100 ps).
The central Li layers stay fixed. A Langevin thermostat is used because it acts on each atom and keeps the mobile atoms at
the target temperature even with fixed atoms.

[JA]

# 02 — 界面の NVT-MD

`01_build_interface.ipynb` で作った界面について、Langevin NVT-MD を実行します（既定：100,000 ステップ、300 K、1 fs/ステップ → 100 ps）。
中央の Li 層は固定します。固定原子があっても動く原子を目標温度に保てるよう、原子ごとに作用する Langevin 熱浴を使います。

[EN]

## Imports

[JA]

## ライブラリのインポート

In [ ]:
%load_ext watermark
%watermark -v -p numpy,matplotlib,ase,pfp_api_client

In [ ]:
import json
import os
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from ase import units
from ase.constraints import FixAtoms
from ase.io import Trajectory, read
from ase.md import MDLogger
from ase.md.langevin import Langevin
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
from pfp_api_client.pfp.calculators.ase_calculator import ASECalculator
from pfp_api_client.pfp.estimator import Estimator, EstimatorCalcMode

[EN]

## Parameters

The model (`model_version`, `calc_mode`) is read from `system_info.json` of `01_build_interface.ipynb`.
Set `quick_test = True` for a short test (200 steps).

[JA]

## パラメータ

モデル（`model_version`、`calc_mode`）は、`01_build_interface.ipynb` の `system_info.json` から読み込みます。
短い動作確認（200 ステップ）をする場合は `quick_test = True` にしてください。

In [ ]:
dir_path = Path.cwd()
input_dir = dir_path / "data" / "01_build_interface"
output_dir = dir_path / "data" / "02_run_md"

interface_file = input_dir / "interface_minimized.traj"  # from 01_build_interface.ipynb
system_info_file = input_dir / "system_info.json"        # from 01_build_interface.ipynb

temperature_K = 300.0
timestep_fs = 1.0
md_time_ps = 100.0
save_interval_fs = 100.0
friction_per_fs = 0.01   # Langevin friction (1/fs)
seed = 42
n_progress_reports = 20

restart = True
quick_test = False       # True: 0.2 ps test saved as md_quick_test.* (set run_name in 03_analysis.ipynb accordingly)
quick_test_time_ps = 0.2
quick_test_save_interval_fs = 10.0

os.makedirs(output_dir, exist_ok=True)

In [ ]:
if quick_test:
    md_time_ps, save_interval_fs, run_name = quick_test_time_ps, quick_test_save_interval_fs, "md_quick_test"
else:
    run_name = "md_nvt"
n_steps_total = round(md_time_ps * 1000 / timestep_fs)
save_every = round(save_interval_fs / timestep_fs)
report_every = save_every * max(1, n_steps_total // save_every // n_progress_reports)
traj_file = output_dir / f"{run_name}.traj"
log_file = output_dir / f"{run_name}.log"
info_file = output_dir / f"{run_name}_info.json"
thermo_plot_file = output_dir / f"{run_name}_thermo.png"

system_info = json.loads(system_info_file.read_text())
estimator = Estimator(
    model_version=system_info["model_version"],
    calc_mode=EstimatorCalcMode[system_info["calc_mode"]],
    max_retries=15,
)
calculator = ASECalculator(estimator)
print(f"{run_name}: {n_steps_total} steps ({md_time_ps} ps)")

[EN]

## Initial state

A new run starts from the minimized interface with Maxwell–Boltzmann velocities. With `restart = True` and an existing
trajectory, the run continues from its last frame.

[JA]

## 初期状態

新しい計算は、緩和済みの界面に Maxwell–Boltzmann 分布の速度を与えて始めます。`restart = True` で既存の軌跡がある場合は、その最後のフレームから再開します。

In [ ]:
steps_done = 0
if restart and traj_file.exists():
    with Trajectory(traj_file) as saved:
        steps_done = max(0, (len(saved) - 1) * save_every)
if steps_done > 0:
    atoms = read(traj_file, index=-1)
    print(f"Restarting at step {steps_done}")
else:
    atoms = read(interface_file)
    MaxwellBoltzmannDistribution(atoms, temperature_K=temperature_K, rng=np.random.default_rng(seed))

atoms.set_constraint(FixAtoms(indices=system_info["fixed_indices"]))
atoms.calc = calculator

[EN]

## Run

Positions, momenta, energies and atomic charges are saved every `save_interval_fs`. 

[JA]

## 実行

`save_interval_fs` ごとに、座標・運動量・エネルギー・原子電荷を保存します。

In [ ]:
dyn = Langevin(
    atoms,
    timestep=timestep_fs * units.fs,
    temperature_K=temperature_K,
    friction=friction_per_fs / units.fs,
    fixcm=False,
    rng=np.random.default_rng(seed + steps_done),  # new random numbers after a restart
)
dyn.nsteps = steps_done  # continue the step count after a restart
traj = Trajectory(traj_file, "a" if steps_done else "w", atoms)


def write_frame():
    atoms.get_charges()  # make sure the atomic charges are stored in the trajectory
    traj.write(atoms)


t_start = time.perf_counter()


def report():
    elapsed = time.perf_counter() - t_start
    remaining = elapsed / max(dyn.nsteps - steps_done, 1) * (n_steps_total - dyn.nsteps)
    print(f"step {dyn.nsteps:7d}  T = {atoms.get_temperature():6.1f} K  "
          f"Epot = {atoms.get_potential_energy() / len(atoms):8.4f} eV/atom  "
          f"elapsed {elapsed / 60:6.1f} min  remaining ~{remaining / 60:6.1f} min", flush=True)


dyn.attach(write_frame, interval=save_every)
dyn.attach(MDLogger(dyn, atoms, str(log_file), header=steps_done == 0, peratom=True,
                    mode="a" if steps_done else "w"), interval=save_every)
dyn.attach(report, interval=report_every)
dyn.run(n_steps_total - steps_done)
traj.close()
print(f"Finished: {(time.perf_counter() - t_start) / 60:.1f} min")

In [ ]:
run_info = {
    "run_name": run_name,
    "trajectory": traj_file.name,
    "log": log_file.name,
    "temperature_K": temperature_K,
    "timestep_fs": timestep_fs,
    "save_interval_fs": save_every * timestep_fs,
    "friction_per_fs": friction_per_fs,
    "n_steps": int(dyn.nsteps),
}
info_file.write_text(json.dumps(run_info, indent=1))

[EN]

## Temperature and energy

The temperature should fluctuate around the target. 

[JA]

## 温度とエネルギー


In [ ]:
log = np.loadtxt(log_file, skiprows=1)
time_ps, epot, temperature = log[:, 0], log[:, 2], log[:, 4]

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(time_ps, temperature)
axes[0].axhline(temperature_K, color="gray", ls="--", lw=1)
axes[0].set_ylabel("Temperature (K)")
axes[1].plot(time_ps, epot)
axes[1].set_ylabel("Potential energy (eV/atom)")
for ax in axes:
    ax.set_xlabel("Time (ps)")
fig.tight_layout()
fig.savefig(thermo_plot_file, dpi=150)
plt.show()
later = time_ps > min(1.0, time_ps[-1] / 2)  # skip the initial temperature drop
print(f"Mean temperature: {temperature[later].mean():.1f} K")

[ALL]

Copyright 2026 Matlantis Corp. as contributors to Matlantis contrib project (supported by Tata Consultancy services Ltd.)